# Libs

In [1]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
# try lib polars
import polars as pl

In [2]:
'''  
Pré-processamento sugerido:

Remoção de valores ambíguos como "Don’t know", "Refused to answer".

Eliminação de colunas com mais de 30% de valores faltantes.

Substituição de códigos como 88/888 por zero em variáveis de contagem de dias.

Aplicação de normalização MinMaxScaler.

Redução de colinearidade com análise de correlação (Pearson).

Binning em variáveis contínuas como altura e peso.

'''

'  \nPré-processamento sugerido:\n\nRemoção de valores ambíguos como "Don’t know", "Refused to answer".\n\nEliminação de colunas com mais de 30% de valores faltantes.\n\nSubstituição de códigos como 88/888 por zero em variáveis de contagem de dias.\n\nAplicação de normalização MinMaxScaler.\n\nRedução de colinearidade com análise de correlação (Pearson).\n\nBinning em variáveis contínuas como altura e peso.\n\n'

In [3]:
# mapeando o diretório do projeto e do arquivo notebook 
diretorio_atual_projeto = os.getcwd() # Diretório atual do arquivo
notebook_dir_project_predict = os.path.normpath(f"{diretorio_atual_projeto}{os.sep}..{os.sep}..{os.sep}") + os.sep # Diretório do projeto
print(diretorio_atual_projeto)
print(notebook_dir_project_predict)

c:\Users\USER\OneDrive\Documentos\proj_tcc_dsa\riskPredictionDeseases\dev\notebooks
c:\Users\USER\OneDrive\Documentos\proj_tcc_dsa\riskPredictionDeseases\


# Functions

In [4]:
# mapear o DataFrame de acordo com o codebook 

def mapear_colunas_para_rotulo(df_brfss, codebook_df, sufixo='_map'):
    """
    Substitui colunas do DataFrame por versões mapeadas com rótulos do codebook.
    A coluna original é excluída e substituída por uma nova com sufixo (default: _map).
    Se o valor não for mapeável, mantém o valor original.

    Parâmetros:
        df_brfss (pd.DataFrame): dados originais
        codebook_df (pd.DataFrame): DataFrame do codebook com 'SAS Variable Name', 'Value', 'Value Label'
        sufixo (str): sufixo para a nova coluna (default: '_map')

    Retorno:
        pd.DataFrame com colunas mapeadas
    """
    def limpar_valor(v):
        if pd.isna(v):
            return "BLANK"
        try:
            return str(int(v))  # converte 1.0 → '1'
        except:
            return str(v).strip()

    df_resultado = df_brfss.copy()
    colunas_mapeadas = {}

    with tqdm(total=len(df_brfss.columns), desc="Processing columns") as pbar:

        for coluna in df_brfss.columns:
            pbar.update(1)
            try:
                # Extrai mapeamentos apenas para a variável atual
                codebook_var = codebook_df[codebook_df['SAS Variable Name'] == coluna]
                codebook_var = codebook_var.dropna(subset=['Value'])

                if codebook_var.empty:
                    continue

                # Cria o dicionário de mapeamento
                mapa_valores = dict(zip(
                    codebook_var['Value'].astype(str).str.strip(),
                    codebook_var['Value Label']
                ))

                if not mapa_valores:
                    continue

                # Aplica mapeamento apenas onde existir valor no dicionário
                serie_convertida = df_brfss[coluna].apply(limpar_valor)
                serie_mapeada = serie_convertida.apply(lambda x: mapa_valores.get(x, x))  # mantém valor original se não estiver no dicionário
                colunas_mapeadas[coluna + sufixo] = serie_mapeada
            
            except Exception as e:
                print(f"Erro ao mapear coluna '{coluna}': {e}")
                continue

            

    # Cria DataFrame com as colunas mapeadas
    df_mapeadas = pd.DataFrame(colunas_mapeadas)

    # Remove colunas originais que foram mapeadas
    colunas_para_remover = [col.replace(sufixo, '') for col in tqdm(df_mapeadas.columns, desc='Remove suport columns')]
    df_resultado = df_resultado.drop(columns=colunas_para_remover)

    # Junta com colunas mapeadas
    df_resultado = pd.concat([df_mapeadas, df_resultado], axis=1)

    return df_resultado

# lista de colunas onde devemos retirar da string os seguintes caracteres "b'" di começo e "'" no final # IDATE IMONTH IDAY IYEAR
# IDATE IMONTH IDAY IYEAR
def limpar_colunas_data(df):
    """
    Função para limpar as colunas de data do DataFrame
    """
    # Limpando as colunas
    df['IDATE'] = df['IDATE'].str.replace("b'", "").str.replace("'", "")
    df['IMONTH'] = df['IMONTH'].str.replace("b'", "").str.replace("'", "")
    df['SEQNO'] = df['SEQNO'].str.replace("b'", "").str.replace("'", "")
    # df['IDAY'] = df['IDAY'].str.replace("b'", "").str.replace("'", "")
    # df['IYEAR'] = df['IYEAR'].str.replace("b'", "").str.replace("'", "")
    
    return df

# faremos uma lista de colunas que serão ignoradas pois todos os dados são o mesmo valor ou nulo e são dados de identificação de amostra





In [5]:
def read_data_and_codebook(path_csv_data, path_csv_codebook):
    """
    Função para ler os dados e o código  .
    """
    # Lendo os dados
    df = pd.read_csv(path_csv_data)
    
    # Lendo o código
    codebook = pd.read_csv(path_csv_codebook)
    
    return df, codebook

# transformar os dados categorizados como dont know ou refused em NaN
def transform_dont_know_refused_to_nan(df):
    """
    Função para transformar os dados categorizados como dont know ou refused em NaN
    """
    # Transformando os dados categorizados como dont know ou refused em NaN
    df = df.replace({'dont know': np.nan, 'refused': np.nan})
    
    return df

# retirar as colunas com mais de 30% de missing data conforme artigo
def remove_columns_with_missing_data(df, threshold=0.3):
    """
    Função para remover colunas com mais de 30% de missing data
    """
    # Calculando o percentual de missing data
    missing_data = df.isnull().mean()
    
    # Removendo as colunas com mais de 30% de missing data
    df = df.loc[:, missing_data < threshold]
    
    return df

In [6]:
# implementada a substituição por vazio 
def substituir_valores_por_zero_baseado_no_codebook(df_brfss, codebook_df, strings_para_zero, sufixo='_processed'):
    """
    Processa colunas do DataFrame:
    Se o rótulo de um valor no codebook contiver alguma das 'strings_para_zero',
    o valor correspondente no DataFrame de dados é substituído por 0.
    Caso contrário, o valor original no DataFrame de dados é mantido.
    As colunas processadas substituem as originais com um sufixo.

    Parâmetros:
        df_brfss (pd.DataFrame): DataFrame original com os dados.
        codebook_df (pd.DataFrame): DataFrame do codebook com 'SAS Variable Name', 'Value', 'Value Label'.
        strings_para_zero (list): Lista de strings que, se encontradas no 'Value Label'
                                  do codebook, farão com que o valor original no df_brfss
                                  seja substituído por 0.
        sufixo (str): Sufixo para as novas colunas processadas.

    Retorno:
        pd.DataFrame com as colunas processadas.
    """

    def limpar_valor_para_lookup(v):
        """Limpa e converte valor para string para lookup no dicionário do codebook."""
        if pd.isna(v):
            return "INTERNAL_NAN_REPR" # Representação interna para NaNs originais dos dados
        try:
            # Tenta converter para int (para lidar com 1.0 -> '1'), depois para string
            return str(int(float(v)))
        except ValueError:
            # Se não puder ser convertido para float/int, usa como string
            return str(v).strip()
        except Exception:
            return str(v).strip() # Fallback

    df_processado = df_brfss.copy()
    colunas_originais_para_remover = []

    strings_para_zero_lower = [s.lower() for s in strings_para_zero]

    with tqdm(total=len(df_processado.columns), desc="Processando colunas") as pbar:
        for coluna in df_processado.columns:
            pbar.update(1)
            
            # Pega as entradas do codebook para a coluna atual
            codebook_var_atual = codebook_df[codebook_df['SAS Variable Name'] == coluna]
            
            if codebook_var_atual.empty:
                continue # Pula para a próxima coluna se não houver info no codebook

            # Cria um mapa de código (Value) para rótulo (Value Label)
            # Limpa os 'Value' do codebook para string para consistência
            mapa_codigo_rotulo = dict(zip(
                codebook_var_atual['Value'].apply(limpar_valor_para_lookup),
                codebook_var_atual['Value Label']
            ))
            
            # Série original da coluna a ser processada
            serie_original = df_processado[coluna].copy()
            # Série que será modificada (começa como uma cópia)
            serie_modificada = df_processado[coluna].copy()

            for idx, valor_original_na_serie in serie_original.items():
                valor_limpo_dados = limpar_valor_para_lookup(valor_original_na_serie)
                
                # Pega o rótulo do codebook para o valor limpo dos dados
                rotulo_do_codebook = mapa_codigo_rotulo.get(valor_limpo_dados)

                if rotulo_do_codebook: # Se encontrou um rótulo no codebook
                    # Verifica se alguma das strings_para_zero está no rótulo
                    if any(s_lower in str(rotulo_do_codebook).lower() for s_lower in strings_para_zero_lower):
                        serie_modificada.loc[idx] = np.nan # 0
                    # else: o valor original já está em serie_modificada, então não faz nada
                elif valor_limpo_dados == "INTERNAL_NAN_REPR" and "blank" in strings_para_zero_lower:
                    # Trata NaNs originais nos dados se "blank" for uma string para zerar
                    serie_modificada.loc[idx] = np.nan # 0
                # else: valor não encontrado no codebook ou rótulo não corresponde, mantém original

            # Atualiza a coluna no DataFrame processado
            df_processado[coluna + sufixo] = serie_modificada
            if sufixo : # Adiciona à lista para remover depois, apenas se houver sufixo
                colunas_originais_para_remover.append(coluna)
    
    # Remove as colunas originais que foram processadas (se o sufixo for diferente de vazio)
    if sufixo and colunas_originais_para_remover:
        colunas_existentes_para_remover = [col for col in colunas_originais_para_remover if col in df_processado.columns]
        df_processado.drop(columns=colunas_existentes_para_remover, inplace=True)
        
    return df_processado

In [7]:
# remoção de strings indesejadas

def limpar_valores_indesejados_codebook(df_brfss, codebook_df, rotulos_invalidos):
    """
    Substitui por NaN os valores do DataFrame que correspondem a rótulos inválidos do codebook.

    Parâmetros:
        df_brfss (pd.DataFrame): dados originais com valores numéricos
        codebook_df (pd.DataFrame): codebook com colunas 'SAS Variable Name', 'Value', 'Value Label'
        rotulos_invalidos (list): lista de strings com rótulos que devem ser tratados como NaN

    Retorno:
        pd.DataFrame com valores substituídos por NaN onde os rótulos são inválidos
    """
    df_resultado = df_brfss.copy()

    for coluna in df_resultado.columns:
        try:
            codebook_var = codebook_df[codebook_df['SAS Variable Name'] == coluna]
            codebook_var = codebook_var.dropna(subset=['Value', 'Value Label'])

            # Filtra os valores que têm rótulos indesejados
            valores_invalidos = codebook_var[
                codebook_var['Value Label'].str.strip().isin(rotulos_invalidos)
            ]['Value']

            # Converte para float para comparar com os dados
            valores_invalidos_float = valores_invalidos.astype(float).tolist()

            # Substitui no dado
            df_resultado[coluna] = df_resultado[coluna].apply(
                lambda x: np.nan if x in valores_invalidos_float else x
            )

        except Exception as e:
            print(f"Erro ao processar coluna '{coluna}': {e}")
            continue

    return df_resultado


In [8]:
# retirar as linhas onde a coluna MICHD for nulo

def remover_linhas_com_alvo_nulo(df, nome_coluna_alvo):
    """
    Remove linhas de um DataFrame onde a coluna alvo especificada é nula (NaN).

    Parâmetros:
        df (pd.DataFrame): DataFrame de entrada.
        nome_coluna_alvo (str): Nome da coluna alvo para verificar valores nulos.

    Retorno:
        pd.DataFrame: DataFrame com as linhas nulas na coluna alvo removidas.
    """
    if nome_coluna_alvo not in df.columns:
        print(f"Erro: A coluna '{nome_coluna_alvo}' não existe no DataFrame.")
        return df # Retorna o DataFrame original se a coluna não existir

    linhas_antes = len(df)
    df_processado = df.dropna(subset=[nome_coluna_alvo])
    linhas_depois = len(df_processado)
    
    print(f"Coluna alvo para remoção de nulos: '{nome_coluna_alvo}'")
    print(f"Linhas antes da remoção: {linhas_antes}")
    print(f"Linhas removidas: {linhas_antes - linhas_depois}")
    print(f"Linhas após a remoção: {linhas_depois}")
    
    return df_processado

# Scripts

Ler o codebook e o respectivo dado

In [9]:
'''       
No codebook,
SAS Variable Name - Coluna do Dado bruto 
Value - Valor no dado bruto
Value Label - Descrição do valor no dado bruto

'''
# precisaremos implementar a metodologia descrita no artigo gerando assim um primeiro dataset para treinamento
# ler o csv do dado bruto e do codebook para efetuar as transformações
raw_data_brfss = os.path.normpath(f"{notebook_dir_project_predict}{os.sep}data{os.sep}intermediate{os.sep}2023{os.sep}brfss_2023.csv")
codebook_file = os.path.normpath(f"{notebook_dir_project_predict}{os.sep}data{os.sep}intermediate{os.sep}2023{os.sep}brfss_2023_variaveis_expandidas_translated.csv")

df_brfss_2023 , df_codebook_2023 = read_data_and_codebook(raw_data_brfss, codebook_file)


Remover colunas não documentadas no codebook

Remover colunas identificadas como irrelevantes para o estudo


In [10]:
df_brfss_2023_to_drop = df_brfss_2023.copy()

# detecção das colunas que não estão no codebook
colunas_nao_mapeadas = [col for col in df_brfss_2023_to_drop.columns if col not in df_codebook_2023['SAS Variable Name'].values]

# as colunas a seguir possuem dados de identificação de amostra 
# e não são relevantes para o modelo
irrelevant_columns = [
   'FMONTH', 
   'IDATE', 
   'IMONTH', 
   'IDAY', 
   'IYEAR', 
   'DISPCODE', 
   'SEQNO', 
   '_PSU', 
   'CTELENM1', 
   'CELPHON1',
   'CTELNUM1',
   'CELLFON5',
   'QSTVER',
   ]

df_2023_removed_columns = df_brfss_2023_to_drop.drop(columns=(colunas_nao_mapeadas + irrelevant_columns), errors='ignore')
# df_2023_removed_columns


In [11]:
'''  
Coluna: NUMADULT
Seráa deletada por ter 80% de missing data

'''

# precisaremos remover alguns valores cuidadosamente 
# que não estão devidamente docuentados no codebook 
# e estão no dado bruto



values_to_remove_not_in_codebook = {

 'LANDSEX2': [3],
 'CELLSEX2': [3],
 'CCLGHOUS': [2],

}

df_2023_to_remove_values  = df_2023_removed_columns.copy()

# ler a coluna na chave do dicionário e remover os valores no dado
for column, values in values_to_remove_not_in_codebook.items():
    if column in df_2023_to_remove_values.columns:
        df_2023_to_remove_values = df_2023_to_remove_values[~df_2023_to_remove_values[column].isin(values)]
    else:
        print(f"Coluna {column} não encontrada no DataFrame.")


df_2023_to_remove_values

,_STATE,PVTRESD1,COLGHOUS,STATERE1,LADULT1,NUMADULT,RESPSLC1,LANDSEX2,SAFETIME,CADULT1,...,DROCDY4_,_RFBING6,_DRNKWK2,_RFDRHV8,_FLSHOT7,_PNEUMO3,_AIDTST4,_RFSEAT2,_RFSEAT3,_DRNKDRV
0,1.0,1.0,NaN,1.0,1.0,2.0,1.0,2.0,NaN,NaN,...,5.397605e-79,1.0,5.397605e-79,1.0,2.0,2.0,2.0,1.0,1.0,9.0
1,1.0,1.0,NaN,1.0,1.0,1.0,NaN,2.0,NaN,NaN,...,5.397605e-79,1.0,5.397605e-79,1.0,1.0,1.0,2.0,1.0,1.0,9.0
2,1.0,1.0,NaN,1.0,1.0,1.0,NaN,2.0,NaN,NaN,...,5.397605e-79,1.0,5.397605e-79,1.0,1.0,1.0,2.0,1.0,1.0,9.0
3,1.0,1.0,NaN,1.0,1.0,2.0,1.0,2.0,NaN,NaN,...,5.397605e-79,1.0,5.397605e-79,1.0,1.0,1.0,1.0,1.0,1.0,9.0
4,1.0,1.0,NaN,1.0,1.0,1.0,NaN,2.0,NaN,NaN,...,7.000000e+00,1.0,4.700000e+01,1.0,2.0,1.0,2.0,1.0,1.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
433318,78.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,...,7.100000e+01,2.0,1.500000e+03,2.0,2.0,2.0,1.0,1.0,2.0,2.0
433319,78.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,...,5.397605e-79,1.0,5.397605e-79,1.0,NaN,NaN,1.0,1.0,1.0,9.0
433320,78.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,...,3.000000e+00,1.0,4.700000e+01,1.0,NaN,NaN,1.0,1.0,1.0,2.0
433321,78.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,...,5.397605e-79,1.0,5.397605e-79,1.0,2.0,2.0,2.0,1.0,1.0,9.0


In [ ]:
# verificar os valor que estão como dias , vezs por semana, meses, anos e etc
# Norrmalzar certos dados 



In [12]:
# remover do dado os valores que se referem a strings inválidas no codebook 

# Quando o valor no codebook for algum desses da lista , mudaremos para vazio
df_remove_string_index = df_2023_to_remove_values.copy()

string_list_to_null = [
 "Refused",
 "Don’t know",
 "Not sure",
 "Don’t know/Not sure",
 "Refused to answer",
 "Blank",
 "Missing",
 "Not asked or Missing",
 "None"
 ]

# df_withhout_invalid = limpar_valores_indesejados_codebook(df_remove_string_index, codebook_2023, string_list_to_null)
# alteramos o argumento para substituir por vazio
df_withhout_invalid = substituir_valores_por_zero_baseado_no_codebook(df_remove_string_index, df_codebook_2023, string_list_to_null, sufixo='_processed')
df_withhout_invalid.to_csv(f"{notebook_dir_project_predict}{os.sep}data{os.sep}intermediate{os.sep}2023{os.sep}brfss_2023_string_null.csv", index=False)
df_withhout_invalid


Processando colunas:  30%|███       | 100/330 [09:21<49:59, 13.04s/it]C:\Users\USER\AppData\Local\Temp\ipykernel_4492\3386865788.py:79: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_processado[coluna + sufixo] = serie_modificada
Processando colunas:  31%|███       | 101/330 [09:35<50:43, 13.29s/it]C:\Users\USER\AppData\Local\Temp\ipykernel_4492\3386865788.py:79: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_processado[coluna + sufixo] = serie_modificada
Processando colunas:  31%|███       | 102/330 [09:49<50:48, 13.37s/it]

,_STATE_processed,PVTRESD1_processed,COLGHOUS_processed,STATERE1_processed,LADULT1_processed,NUMADULT_processed,RESPSLC1_processed,LANDSEX2_processed,SAFETIME_processed,CADULT1_processed,...,DROCDY4__processed,_RFBING6_processed,_DRNKWK2_processed,_RFDRHV8_processed,_FLSHOT7_processed,_PNEUMO3_processed,_AIDTST4_processed,_RFSEAT2_processed,_RFSEAT3_processed,_DRNKDRV_processed
0,1.0,1.0,NaN,1.0,1.0,2.0,1.0,2.0,NaN,NaN,...,5.397605e-79,1.0,5.397605e-79,1.0,2.0,2.0,2.0,1.0,1.0,NaN
1,1.0,1.0,NaN,1.0,1.0,1.0,NaN,2.0,NaN,NaN,...,5.397605e-79,1.0,5.397605e-79,1.0,1.0,1.0,2.0,1.0,1.0,NaN
2,1.0,1.0,NaN,1.0,1.0,1.0,NaN,2.0,NaN,NaN,...,5.397605e-79,1.0,5.397605e-79,1.0,1.0,1.0,2.0,1.0,1.0,NaN
3,1.0,1.0,NaN,1.0,1.0,2.0,1.0,2.0,NaN,NaN,...,5.397605e-79,1.0,5.397605e-79,1.0,1.0,1.0,1.0,1.0,1.0,NaN
4,1.0,1.0,NaN,1.0,1.0,1.0,NaN,2.0,NaN,NaN,...,7.000000e+00,1.0,4.700000e+01,1.0,2.0,1.0,2.0,1.0,1.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
433318,78.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,...,7.100000e+01,2.0,1.500000e+03,2.0,2.0,2.0,1.0,1.0,2.0,2.0
433319,78.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,...,5.397605e-79,1.0,5.397605e-79,1.0,NaN,NaN,1.0,1.0,1.0,NaN
433320,78.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,...,3.000000e+00,1.0,4.700000e+01,1.0,NaN,NaN,1.0,1.0,1.0,2.0
433321,78.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,...,5.397605e-79,1.0,5.397605e-79,1.0,2.0,2.0,2.0,1.0,1.0,NaN


In [ ]:
'''
Recaptulando
Variáveis de interesse 
CVDINFR4 = Ever Diagnosed with Heart Attack 
CVDCRHD4 = Ever Diagnosed with Angina or Coronary Heart Disease 
_MICHD = Ever Diagnosed with Heart Disease 

'''

# assim vamos retirar os registros (linhas) onde é vazio para a variável MICHD
# removar o missing data por linha , todos que foem nulos para as variável MICHD
'''
df_remove_null_lines = df_withhout_invalid.copy()
df_remove_null_lines_result = remover_linhas_com_alvo_nulo(df_remove_null_lines, '_MICHD_processed')
df_remove_null_lines_result.to_csv(f"{notebook_dir_project_predict}{os.sep}data{os.sep}intermediate{os.sep}2023{os.sep}brfss_2023_michd_not_null.csv", index=False)
df_remove_null_lines_result'''

Coluna alvo para remoção de nulos: '_MICHD_processed'
Linhas antes da remoção: 431042
Linhas removidas: 4560
Linhas após a remoção: 426482


,_STATE_processed,PVTRESD1_processed,COLGHOUS_processed,STATERE1_processed,LADULT1_processed,NUMADULT_processed,RESPSLC1_processed,LANDSEX2_processed,SAFETIME_processed,CADULT1_processed,...,DROCDY4__processed,_RFBING6_processed,_DRNKWK2_processed,_RFDRHV8_processed,_FLSHOT7_processed,_PNEUMO3_processed,_AIDTST4_processed,_RFSEAT2_processed,_RFSEAT3_processed,_DRNKDRV_processed
0,1.0,1.0,NaN,1.0,1.0,2.0,1.0,2.0,NaN,NaN,...,5.397605e-79,1.0,5.397605e-79,1.0,2.0,2.0,2.0,1.0,1.0,NaN
1,1.0,1.0,NaN,1.0,1.0,1.0,NaN,2.0,NaN,NaN,...,5.397605e-79,1.0,5.397605e-79,1.0,1.0,1.0,2.0,1.0,1.0,NaN
2,1.0,1.0,NaN,1.0,1.0,1.0,NaN,2.0,NaN,NaN,...,5.397605e-79,1.0,5.397605e-79,1.0,1.0,1.0,2.0,1.0,1.0,NaN
3,1.0,1.0,NaN,1.0,1.0,2.0,1.0,2.0,NaN,NaN,...,5.397605e-79,1.0,5.397605e-79,1.0,1.0,1.0,1.0,1.0,1.0,NaN
4,1.0,1.0,NaN,1.0,1.0,1.0,NaN,2.0,NaN,NaN,...,7.000000e+00,1.0,4.700000e+01,1.0,2.0,1.0,2.0,1.0,1.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
433318,78.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,...,7.100000e+01,2.0,1.500000e+03,2.0,2.0,2.0,1.0,1.0,2.0,2.0
433319,78.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,...,5.397605e-79,1.0,5.397605e-79,1.0,NaN,NaN,1.0,1.0,1.0,NaN
433320,78.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,...,3.000000e+00,1.0,4.700000e+01,1.0,NaN,NaN,1.0,1.0,1.0,2.0
433321,78.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,...,5.397605e-79,1.0,5.397605e-79,1.0,2.0,2.0,2.0,1.0,1.0,NaN


In [29]:
# Aplicar seleção de variáveis com menos de 25 % de missing após filtrar o Heart Attack and Stroke
# df_brfss_2023_filtered_columns_csv = pd.read_csv(f"{notebook_dir_project_predict}data{os.sep}intermediate{os.sep}2023{os.sep}brfss_2023.csv")
df_brfss_2023_filtered_columns_csv = df_withhout_invalid.copy()

# Calcular o percentual de valores nulos por coluna
percentual_na = df_brfss_2023_filtered_columns_csv.isnull().mean() * 100

# Selecionar colunas com menos de 30% de valores nulos - usar < 30 # conforme o paper
colunas_boas = percentual_na[percentual_na  < 30].index # Usar a proporção de missing data da variavel alvo para filtrar as colunas

# Criar novo DataFrame apenas com essas colunas
df_brfss_2023_csv_filtered = df_brfss_2023_filtered_columns_csv[colunas_boas]

# Exibir informações do DataFrame filtrado
df_brfss_2023_csv_filtered.info()

# Exibir o DataFrame
df_brfss_2023_csv_filtered

<class 'pandas.core.frame.DataFrame'>
Index: 431042 entries, 0 to 433322
Columns: 138 entries, _STATE_processed to _RFSEAT3_processed
dtypes: float64(138)
memory usage: 473.2 MB


,_STATE_processed,SAFETIME_processed,CADULT1_processed,CELLSEX2_processed,PVTRESD3_processed,CSTATE1_processed,LANDLINE_processed,HHADULT_processed,SEXVAR_processed,GENHLTH_processed,...,_RFSMOK3_processed,_CURECI2_processed,DRNKANY6_processed,DROCDY4__processed,_RFBING6_processed,_DRNKWK2_processed,_RFDRHV8_processed,_AIDTST4_processed,_RFSEAT2_processed,_RFSEAT3_processed
0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,2.0,...,1.0,1.0,2.0,5.397605e-79,1.0,5.397605e-79,1.0,2.0,1.0,1.0
1,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,2.0,...,1.0,1.0,2.0,5.397605e-79,1.0,5.397605e-79,1.0,2.0,1.0,1.0
2,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,4.0,...,1.0,1.0,2.0,5.397605e-79,1.0,5.397605e-79,1.0,2.0,1.0,1.0
3,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,2.0,...,1.0,1.0,2.0,5.397605e-79,1.0,5.397605e-79,1.0,1.0,1.0,1.0
4,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,4.0,...,1.0,1.0,1.0,7.000000e+00,1.0,4.700000e+01,1.0,2.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
433318,78.0,1.0,1.0,1.0,1.0,1.0,2.0,1.0,1.0,3.0,...,1.0,1.0,1.0,7.100000e+01,2.0,1.500000e+03,2.0,1.0,1.0,2.0
433319,78.0,1.0,1.0,2.0,1.0,1.0,2.0,3.0,2.0,2.0,...,1.0,1.0,2.0,5.397605e-79,1.0,5.397605e-79,1.0,1.0,1.0,1.0
433320,78.0,1.0,1.0,2.0,1.0,1.0,2.0,4.0,2.0,2.0,...,1.0,1.0,1.0,3.000000e+00,1.0,4.700000e+01,1.0,1.0,1.0,1.0
433321,78.0,1.0,1.0,2.0,1.0,1.0,2.0,1.0,2.0,3.0,...,1.0,1.0,2.0,5.397605e-79,1.0,5.397605e-79,1.0,2.0,1.0,1.0


Removendo o missing data por linha , deixando apenas os registros totalmente completos

In [27]:
'''  
Outros tratamentos de strings e dados de data
Tratamento de missing data (30% conforme o paper)
Tratamento de variáveis contínuas

'''
# a amostra onde a linha tiver mais de 30% de missing data


def remover_linhas_com_muitos_nulos(df, limiar_percentual_nulos_linha=0):
    """
    Remove linhas de um DataFrame que excedem um determinado limiar
    percentual de valores ausentes (NaN).

    Parâmetros:
        df (pd.DataFrame): DataFrame de entrada.
        limiar_percentual_nulos_linha (float): Limiar percentual (0-100).
                                                Linhas com mais % de NaNs do que
                                                este valor serão removidas.
                                                Default é 30.0 (30%).

    Retorno:
        pd.DataFrame: DataFrame com as linhas problemáticas removidas.
    """
    if not isinstance(df, pd.DataFrame):
        raise ValueError("A entrada 'df' deve ser um DataFrame Pandas.")
    if not (0 <= limiar_percentual_nulos_linha <= 100):
        raise ValueError("O 'limiar_percentual_nulos_linha' deve estar entre 0 e 100.")

    print(f"DataFrame original - Shape: {df.shape}")

    # Calcula o número mínimo de valores não nulos que uma linha deve ter
    # Se uma linha tiver menos que isso, ela tem mais do que o limiar de nulos
    min_valores_nao_nulos_por_linha = int(df.shape[1] * (1 - (limiar_percentual_nulos_linha / 100.0)))
    
    # Garante que o mínimo não seja negativo se o limiar for 100%
    min_valores_nao_nulos_por_linha = max(0, min_valores_nao_nulos_por_linha)

    print(f"Limiar para remoção: Linhas com mais de {limiar_percentual_nulos_linha}% de valores nulos.")
    print(f"Isso significa que uma linha deve ter pelo menos {min_valores_nao_nulos_por_linha} valores não nulos (de {df.shape[1]} colunas).")

    # df.dropna() com o parâmetro 'thresh' mantém linhas com pelo menos 'thresh' valores não nulos.
    df_filtrado = df.dropna(thresh=min_valores_nao_nulos_por_linha, axis=0) # axis=0 para operar nas linhas

    print(f"DataFrame após remoção de linhas - Shape: {df_filtrado.shape}")
    print(f"Número de linhas removidas: {df.shape[0] - df_filtrado.shape[0]}")
    
    return df_filtrado



In [32]:
df_to_filter_samples = df_brfss_2023_csv_filtered.copy()

df_sem_linhas_com_muitos_nulos = remover_linhas_com_muitos_nulos(df_to_filter_samples, 
                                                                 limiar_percentual_nulos_linha=0)

df_sem_linhas_com_muitos_nulos.to_csv(f"{notebook_dir_project_predict}{os.sep}data{os.sep}output{os.sep}2023{os.sep}brfss_2023_cleaned_to_model.csv", index=False)

df_sem_linhas_com_muitos_nulos                            


DataFrame original - Shape: (431042, 138)
Limiar para remoção: Linhas com mais de 0% de valores nulos.
Isso significa que uma linha deve ter pelo menos 138 valores não nulos (de 138 colunas).
DataFrame após remoção de linhas - Shape: (99799, 138)
Número de linhas removidas: 331243


,_STATE_processed,SAFETIME_processed,CADULT1_processed,CELLSEX2_processed,PVTRESD3_processed,CSTATE1_processed,LANDLINE_processed,HHADULT_processed,SEXVAR_processed,GENHLTH_processed,...,_RFSMOK3_processed,_CURECI2_processed,DRNKANY6_processed,DROCDY4__processed,_RFBING6_processed,_DRNKWK2_processed,_RFDRHV8_processed,_AIDTST4_processed,_RFSEAT2_processed,_RFSEAT3_processed
963,1.0,1.0,1.0,1.0,1.0,1.0,2.0,2.0,1.0,2.0,...,1.0,1.0,1.0,7.000000e+00,1.0,4.700000e+01,1.0,1.0,1.0,1.0
971,1.0,1.0,1.0,1.0,1.0,1.0,1.0,2.0,1.0,2.0,...,2.0,1.0,1.0,1.300000e+01,2.0,1.120000e+03,1.0,2.0,1.0,1.0
977,1.0,1.0,1.0,2.0,1.0,1.0,2.0,1.0,2.0,3.0,...,1.0,1.0,2.0,5.397605e-79,1.0,5.397605e-79,1.0,2.0,2.0,2.0
980,1.0,1.0,1.0,2.0,1.0,1.0,1.0,1.0,2.0,5.0,...,2.0,1.0,2.0,5.397605e-79,1.0,5.397605e-79,1.0,2.0,1.0,1.0
982,1.0,1.0,1.0,1.0,1.0,1.0,2.0,2.0,1.0,1.0,...,1.0,1.0,2.0,5.397605e-79,1.0,5.397605e-79,1.0,2.0,2.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
425094,56.0,1.0,1.0,2.0,1.0,1.0,2.0,1.0,2.0,3.0,...,2.0,1.0,1.0,1.400000e+01,2.0,4.000000e+02,1.0,2.0,2.0,2.0
425097,56.0,1.0,1.0,2.0,1.0,1.0,2.0,2.0,2.0,3.0,...,1.0,1.0,2.0,5.397605e-79,1.0,5.397605e-79,1.0,2.0,1.0,1.0
425098,56.0,1.0,1.0,2.0,1.0,1.0,2.0,1.0,2.0,1.0,...,1.0,1.0,1.0,7.100000e+01,1.0,5.000000e+02,1.0,2.0,1.0,1.0
425099,56.0,1.0,1.0,1.0,1.0,1.0,2.0,1.0,1.0,5.0,...,1.0,1.0,2.0,5.397605e-79,1.0,5.397605e-79,1.0,1.0,1.0,1.0


In [37]:
# Resultados do dataframe para os modelos
# MICHD 
df_sem_linhas_com_muitos_nulos[[
    # '_MICHD_processed', 
    # 'CVDINFR4_processed', 
    'CVDCRHD4_processed'
]].value_counts().sort_index()



CVDCRHD4_processed
1.0                    4019
2.0                   95780
Name: count, dtype: int64

In [42]:
df_sem_linhas_com_muitos_nulos.shape

(99799, 138)

Detalhes das variáveis pos tratamento

In [9]:
import pandas as pd
import numpy as np

# Certifique-se de que o DataFrame 'df_sem_linhas_com_muitos_nulos' 
# já está carregado e disponível no seu ambiente antes de executar este trecho.
# Exemplo de como você o teria (você precisa ter este DataFrame definido):
# df_sem_linhas_com_muitos_nulos = pd.read_csv('caminho_para_seu_arquivo_processado.csv') 
# OU
# df_sem_linhas_com_muitos_nulos = sua_funcao_de_processamento_anterior()

try:
    # Atribui o seu DataFrame real à variável que o script usará
    df_para_analise = pd.read_csv(f"{notebook_dir_project_predict}{os.sep}data{os.sep}output{os.sep}2023{os.sep}brfss_2023_cleaned_to_model.csv")
    print("Utilizando o DataFrame 'df_sem_linhas_com_muitos_nulos' fornecido.")
except NameError:
    print("ERRO CRÍTICO: DataFrame 'df_sem_linhas_com_muitos_nulos' não foi encontrado no ambiente.")
    print("Por favor, certifique-se de que este DataFrame está carregado e definido antes de executar este script.")
    # Para evitar erros subsequentes, criamos um DataFrame vazio.
    # As operações seguintes não produzirão a tabela esperada se isto acontecer.
    df_para_analise = pd.DataFrame() 
except Exception as e:
    print(f"Ocorreu um erro inesperado ao tentar usar 'df_sem_linhas_com_muitos_nulos': {e}")
    df_para_analise = pd.DataFrame()

# Defina os nomes exatos das colunas no seu DataFrame de origem
# e os nomes que você quer na tabela final
mapa_colunas_originais_para_tabela = {
    '_MICHD_processed': 'MICHD',
    'CVDCRHD4_processed': 'CVDCRHD4',
    'CVDINFR4_processed': 'CVDINFR4'
}

lista_dfs_contagens = []

if not df_para_analise.empty: # Procede apenas se df_para_analise não estiver vazio
    print("Calculando value_counts para cada coluna:")
    for nome_coluna_original, nome_coluna_tabela in mapa_colunas_originais_para_tabela.items():
        if nome_coluna_original in df_para_analise.columns:
            # Converte para numérico, erros 'coerce' transformarão não-numéricos em NaN
            serie_numerica = pd.to_numeric(df_para_analise[nome_coluna_original], errors='coerce')
            
            # Calcula value_counts e transforma em DataFrame
            df_contagem_coluna = serie_numerica.value_counts().to_frame(name=nome_coluna_tabela)
            
            print(f"\nValue counts para '{nome_coluna_original}' (como DataFrame '{nome_coluna_tabela}'):")
            print(df_contagem_coluna)
            
            lista_dfs_contagens.append(df_contagem_coluna)
        else:
            print(f"Aviso: Coluna '{nome_coluna_original}' não encontrada no DataFrame 'df_para_analise'.")
            # Cria um DataFrame vazio com o nome da coluna e os índices esperados para evitar erro no concat
            # e para que a coluna apareça com zeros na tabela final.
            lista_dfs_contagens.append(pd.DataFrame(index=pd.Index([1.0, 2.0], name='Valor_Indice_Temp'), columns=[nome_coluna_tabela]).fillna(0))

    # Concatenar todos os DataFrames de contagens ao longo do eixo das colunas (axis=1)
    if lista_dfs_contagens:
        tabela_final_bruta = pd.concat(lista_dfs_contagens, axis=1)
        
        # Preencher NaNs com 0, caso algum valor (1.0 ou 2.0) não exista em alguma coluna
        # e converte para inteiro
        tabela_final_bruta = tabela_final_bruta.fillna(0).astype(int)

        # Reindexar para garantir que temos as linhas para 1.0 e 2.0, preenchendo com 0 se faltar
        # Usamos floats aqui porque value_counts em colunas com NaNs ou floats pode gerar índices float
        indices_desejados_float = [1.0, 2.0]
        tabela_final = tabela_final_bruta.reindex(indices_desejados_float).fillna(0)

        # Se os índices originais pudessem ser inteiros (1, 2) e floats (1.0, 2.0) e ambos existissem
        # após o pd.to_numeric, precisaríamos de uma lógica para somá-los.
        # No entanto, pd.to_numeric seguido de value_counts geralmente resulta em um índice consistente (float se havia NaNs/floats, int caso contrário).
        # A reindexação com [1.0, 2.0] já lida com isso de forma mais limpa.

        # Mapear o índice de float para int (1.0 -> 1, 2.0 -> 2) para a exibição final
        tabela_final = tabela_final.rename(index={1.0: 1, 2.0: 2})
        
        # Filtrar apenas pelos índices 1 e 2 se ainda houver outros (pouco provável após reindex)
        tabela_final = tabela_final.loc[[idx for idx in [1, 2] if idx in tabela_final.index]]

        tabela_final.index.name = 'Valor_Indice'
        tabela_final = tabela_final.astype(int) # Garante que as contagens finais sejam inteiras

        print("\n--- Tabela de Resumo das Contagens (Value Counts e Concat) ---")
        print(tabela_final)

    else:
        print("\nNenhuma coluna de contagem foi processada, tabela final não pode ser gerada.")
else:
    print("\nDataFrame de entrada 'df_para_analise' está vazio ou não foi carregado. Tabela não gerada.")



Utilizando o DataFrame 'df_sem_linhas_com_muitos_nulos' fornecido.
Calculando value_counts para cada coluna:

Value counts para '_MICHD_processed' (como DataFrame 'MICHD'):
                  MICHD
_MICHD_processed       
2.0               93868
1.0                5931

Value counts para 'CVDCRHD4_processed' (como DataFrame 'CVDCRHD4'):
                    CVDCRHD4
CVDCRHD4_processed          
2.0                    95780
1.0                     4019

Value counts para 'CVDINFR4_processed' (como DataFrame 'CVDINFR4'):
                    CVDINFR4
CVDINFR4_processed          
2.0                    96187
1.0                     3612

--- Tabela de Resumo das Contagens (Value Counts e Concat) ---
              MICHD  CVDCRHD4  CVDINFR4
Valor_Indice                           
1              5931      4019      3612
2             93868     95780     96187


In [16]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import os # Para manipulação de caminhos de arquivo

# --- Passo 1: O DataFrame 'tabela_final' DEVE existir neste ponto ---
# Ele é o resultado do script da célula anterior do Jupyter Notebook
# (originalmente do artefato 'tabela_resumo_counts_michd').

# Verifica se 'tabela_final' existe e não está vazia
if 'tabela_final' not in globals() or not isinstance(tabela_final, pd.DataFrame) or tabela_final.empty:
    print("ERRO CRÍTICO: A variável 'tabela_final' não foi encontrada, não é um DataFrame ou está vazia.")
    print("Certifique-se de que a célula anterior que cria 'tabela_final' foi executada com sucesso.")
    tabela_final_valida = False # Flag para controlar a geração do gráfico
else:
    print("Utilizando a variável 'tabela_final' existente para gerar o gráfico.")
    tabela_final_valida = True


if tabela_final_valida:
    # --- Passo 2: Preparar os dados para o Plotly (formato longo) ---
    df_para_plotar = tabela_final.reset_index().melt(
        id_vars='Valor_Indice', 
        var_name='Variavel', 
        value_name='Contagem'
    )
    
    # Mapear Valor_Indice para os rótulos desejados (Positivo/Negativo)
    df_para_plotar['Valor_Indice'] = df_para_plotar['Valor_Indice'].astype(str)
    mapa_rotulos = {
        '1': '1: Positivo', 
        '2': '2: Negativo'  
    }
    df_para_plotar['Status_Caso'] = df_para_plotar['Valor_Indice'].map(mapa_rotulos)
    
    print("\nDataFrame preparado para o gráfico Plotly com Status_Caso:")
    print(df_para_plotar.head())

    # --- Passo 3: Criar o gráfico de barras agrupadas ---
    fig = px.bar(
        df_para_plotar,
        x='Variavel',
        y='Contagem',
        color='Status_Caso', 
        barmode='group', 
        title='Contagem de Casos por Variável (Positivo vs. Negativo)',
        labels={
            'Variavel': 'Variáveis de Diagnóstico', 
            'Contagem': 'Número de Casos', 
            'Status_Caso': 'Status do Caso'
        },
        color_discrete_map={ 
            '1: Positivo': 'red',    
            '2: Negativo': 'blue'   
        }
        # text_auto=True foi removido para usarmos texttemplate para mais controle
    )

    # Ajustes para o texto nas barras e layout
    fig.update_traces(
        texttemplate='%{y:..0f}', # Formato do número: inteiro com separador de milhar (vírgula)
        textposition='outside'
    )
    
    fig.update_layout(
        xaxis_title="Variáveis de Diagnóstico",
        yaxis_title="Número de Casos",
        legend_title_text='Status do Caso', 
        title_x=0.5, # Centralizar o título
        uniformtext_minsize=8, # Garante que o texto caiba
        uniformtext_mode='hide', # Esconde o texto se não couber (opcional)
        margin=dict(t=80, b=50, l=50, r=50) # Aumenta a margem superior (t=top) para dar espaço ao texto
    )
    
    # Para garantir que o eixo Y se estenda o suficiente para o texto no topo:
    # Encontrar o valor máximo para ajustar o range do eixo Y
    max_y_valor = df_para_plotar['Contagem'].max()
    fig.update_yaxes(range=[0, max_y_valor * 1.15]) # Aumenta o limite superior do eixo Y em 15%


    # --- Passo 4: Exibir e/ou Salvar o gráfico ---
    print("\nExibindo o gráfico Plotly...")
    fig.show() 

    caminho_graficos = "graficos_gerados" 
    if not os.path.exists(caminho_graficos):
        os.makedirs(caminho_graficos)
    
    nome_arquivo_html = os.path.join(caminho_graficos, "contagens_status_casos_diagnostico.html") 
    fig.write_html(nome_arquivo_html)
    print(f"Gráfico Plotly também salvo como HTML em: {nome_arquivo_html}")

else:
    print("\n'tabela_final' não está válida ou não foi definida corretamente na célula anterior. Gráfico não pode ser gerado.")



Utilizando a variável 'tabela_final' existente para gerar o gráfico.

DataFrame preparado para o gráfico Plotly com Status_Caso:
  Valor_Indice  Variavel  Contagem  Status_Caso
0            1     MICHD      5931  1: Positivo
1            2     MICHD     93868  2: Negativo
2            1  CVDCRHD4      4019  1: Positivo
3            2  CVDCRHD4     95780  2: Negativo
4            1  CVDINFR4      3612  1: Positivo

Exibindo o gráfico Plotly...


Gráfico Plotly também salvo como HTML em: graficos_gerados\contagens_status_casos_diagnostico.html


Abaixo função para mapear códigos no dado sem referência no codebook

In [30]:


# identificar se nas colunas restantes ignorando o valores vazios tem algum valor não representado no codebook e listar essas colunas e o valor presente no dado que não está no codebook
def encontrar_e_salvar_valores_nao_mapeados(df_brfss, codebook_df, caminho_arquivo_saida):
    """
    Identifica, para cada coluna do DataFrame, os valores que não possuem
    um 'Value Label' correspondente no codebook. Valores nulos (NaN)
    no DataFrame de dados são ignorados. Os resultados são salvos em um arquivo de texto.

    Parâmetros:
        df_brfss (pd.DataFrame): DataFrame original com os dados.
        codebook_df (pd.DataFrame): DataFrame do codebook com 'SAS Variable Name',
                                    'Value', e 'Value Label'.
        caminho_arquivo_saida (str): Caminho completo para o arquivo .txt onde os
                                     resultados serão salvos.

    Retorno:
        dict: Um dicionário onde as chaves são nomes de colunas e os valores
              são listas de valores únicos daquela coluna que não foram
              encontrados nos códigos ('Value') do codebook para aquela variável.
              Retorna apenas colunas que tiveram valores não mapeados.
              Retorna um dicionário vazio se nenhum valor não mapeado for encontrado.
    """

    def limpar_valor_para_comparacao(v):
        """Limpa e converte valor para string para comparação com os códigos do codebook."""
        if pd.isna(v): # Se o valor já for NaN, não há como limpar para string de forma útil aqui.
            return None # Será filtrado depois pelo dropna() nos valores únicos da coluna.
        try:
            return str(int(float(v)))
        except ValueError:
            return str(v).strip()
        except Exception:
            return str(v).strip()

    valores_nao_encontrados_geral = {}

    with tqdm(total=len(df_brfss.columns), desc="Verificando colunas") as pbar:
        for coluna in df_brfss.columns:
            pbar.update(1)
            
            codebook_var_atual = codebook_df[codebook_df['SAS Variable Name'] == coluna]
            
            if codebook_var_atual.empty:
                continue

            codigos_no_codebook_para_coluna = set(
                codebook_var_atual['Value'].dropna().apply(limpar_valor_para_comparacao)
            )

            if not codigos_no_codebook_para_coluna:
                continue
                
            # Pega os valores únicos da coluna no DataFrame de dados, IGNORANDO NaNs
            valores_unicos_na_coluna_dados = df_brfss[coluna].dropna().unique()
            
            valores_nao_encontrados_nesta_coluna = set()

            for valor_dados in valores_unicos_na_coluna_dados:
                valor_dados_limpo = limpar_valor_para_comparacao(valor_dados)
                
                # Se valor_dados_limpo for None (era NaN originalmente), não deve ser comparado
                if valor_dados_limpo is None:
                    continue

                if valor_dados_limpo not in codigos_no_codebook_para_coluna:
                    valores_nao_encontrados_nesta_coluna.add(valor_dados) 
            
            if valores_nao_encontrados_nesta_coluna:
                # Converte para lista e ordena para consistência na saída
                # Garante que os valores sejam strings para evitar problemas de tipo misto no sort
                valores_nao_encontrados_geral[coluna] = sorted(list(map(str, valores_nao_encontrados_nesta_coluna)))
    
    # Salvar os resultados no arquivo de texto
    try:
        with open(caminho_arquivo_saida, 'w', encoding='utf-8') as f:
            if valores_nao_encontrados_geral:
                f.write("Valores não encontrados no codebook (ignorando NaNs nos dados):\n")
                f.write("="*60 + "\n")
                for coluna, valores in valores_nao_encontrados_geral.items():
                    f.write(f"Coluna: {coluna}\n")
                    f.write(f"Valores não mapeados: {valores}\n")
                    f.write("-" * 40 + "\n")
                print(f"\nResultados salvos em: {caminho_arquivo_saida}")
            else:
                f.write("Nenhum valor não mapeado encontrado (ignorando NaNs nos dados).\n")
                print(f"\nNenhum valor não mapeado encontrado. Arquivo salvo em: {caminho_arquivo_saida}")
    except IOError as e:
        print(f"Erro ao salvar o arquivo em '{caminho_arquivo_saida}': {e}")
        # Retorna o dicionário mesmo se o salvamento falhar, para não perder os dados
        return valores_nao_encontrados_geral
    except Exception as e:
        print(f"Ocorreu um erro inesperado ao tentar salvar o arquivo: {e}")
        return valores_nao_encontrados_geral
                
    return valores_nao_encontrados_geral

caminho_arquivo_saida = os.path.normpath(f"{notebook_dir_project_predict}{os.sep}data{os.sep}intermediate{os.sep}2023{os.sep}valores_nao_mapeados.txt")
valores_nao_mapeados = encontrar_e_salvar_valores_nao_mapeados(df_2023_to_remove_values, df_codebook_2023, caminho_arquivo_saida)





Verificando colunas: 100%|██████████| 330/330 [00:00<00:00, 335.58it/s]


Resultados salvos em: /home/ed/lgcm/projects/riskPredictionDeseases/data/intermediate/2023/valores_nao_mapeados.txt


Mapear codebook e gerar um novo dataset

In [ ]:
df_2023_to_treat = df_brfss_2023.copy() # há possívelmente um erro em mapear simplesmente pois alguns valores podem ter mais de uma representação
# df_cleaned_date = limpar_colunas_data(df_2023_to_treat)

df_map = mapear_colunas_para_rotulo(df_2023_to_treat, df_codebook_2023)
df_map.to_csv(f"{notebook_dir_project_predict}{os.sep}data{os.sep}intermediate{os.sep}2023{os.sep}brfss_2023_maped.csv")
df_map

Remove suport columns: 100%|██████████| 343/343 [00:00<00:00, 2573606.93it/s]


,_STATE_map,FMONTH_map,IDATE_map,IMONTH_map,IDAY_map,IYEAR_map,DISPCODE_map,SEQNO_map,_PSU_map,CTELENM1_map,...,_RFSEAT2_map,_RFSEAT3_map,_DRNKDRV_map,LNDSXBRT,CELSXBRT,BIRTHSEX,TRNSGNDR,USEMRJN4,RCSGEND1,RCSXBRTH
0,Alabama,January,b'03012023',b'03',b'01',b'2023',Completed Interview,b'2023000001',2023000001,"Yes - Go to LL.02, PVTRESD1",...,Always or Almost Always Wear Seat Belt,Always Wear Seat Belt,Don't know/Not Sure/Refused/Missing,NaN,NaN,NaN,4.0,NaN,NaN,NaN
1,Alabama,January,b'01062023',b'01',b'06',b'2023',Completed Interview,b'2023000002',2023000002,"Yes - Go to LL.02, PVTRESD1",...,Always or Almost Always Wear Seat Belt,Always Wear Seat Belt,Don't know/Not Sure/Refused/Missing,NaN,NaN,NaN,4.0,NaN,NaN,NaN
2,Alabama,January,b'03082023',b'03',b'08',b'2023',Completed Interview,b'2023000003',2023000003,"Yes - Go to LL.02, PVTRESD1",...,Always or Almost Always Wear Seat Belt,Always Wear Seat Belt,Don't know/Not Sure/Refused/Missing,NaN,NaN,NaN,4.0,NaN,NaN,NaN
3,Alabama,January,b'03062023',b'03',b'06',b'2023',Completed Interview,b'2023000004',2023000004,"Yes - Go to LL.02, PVTRESD1",...,Always or Almost Always Wear Seat Belt,Always Wear Seat Belt,Don't know/Not Sure/Refused/Missing,NaN,NaN,NaN,4.0,NaN,NaN,NaN
4,Alabama,January,b'01062023',b'01',b'06',b'2023',Completed Interview,b'2023000005',2023000005,"Yes - Go to LL.02, PVTRESD1",...,Always or Almost Always Wear Seat Belt,Always Wear Seat Belt,Have not driven after having too much to drink,NaN,NaN,NaN,4.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
433318,Virgin Islands,December,b'12112023',b'12',b'11',b'2023',Completed Interview,b'2023002064',2023002064,MissingNotes: QSTVER > = 20,...,Always or Almost Always Wear Seat Belt,Don't Always Wear Seat Belt,Have not driven after having too much to drink,NaN,NaN,NaN,4.0,NaN,NaN,NaN
433319,Virgin Islands,December,b'01032024',b'01',b'03',b'2024',Completed Interview,b'2023002065',2023002065,MissingNotes: QSTVER > = 20,...,Always or Almost Always Wear Seat Belt,Always Wear Seat Belt,Don't know/Not Sure/Refused/Missing,NaN,NaN,NaN,4.0,NaN,NaN,NaN
433320,Virgin Islands,December,b'12132023',b'12',b'13',b'2023',Completed Interview,b'2023002066',2023002066,MissingNotes: QSTVER > = 20,...,Always or Almost Always Wear Seat Belt,Always Wear Seat Belt,Have not driven after having too much to drink,NaN,NaN,NaN,4.0,NaN,NaN,NaN
433321,Virgin Islands,December,b'12082023',b'12',b'08',b'2023',Completed Interview,b'2023002067',2023002067,MissingNotes: QSTVER > = 20,...,Always or Almost Always Wear Seat Belt,Always Wear Seat Belt,Don't know/Not Sure/Refused/Missing,NaN,NaN,NaN,4.0,NaN,NaN,NaN


Dataset final tratado para uso nos modelos